We began by setting up the usual imports and path constants, then pulled in the two CSVs that form the basis of the analysis (`matched_pairs` and `all_matched_profiles`).

From there the workflow looks like this:

* parse the `counts_by_year` JSON blobs in the profiles table, expanding them to a “long” year‑by‑year
    table and computing a relative year (`rel_year = cal_year – award_year`);
* restrict that long table to the ±10‑year window around the award;
* define pre‑ and post‑award windows (years –5…–1 and +1…+5) and, for each author, compute the
    average citations and works in those windows;
* add a tiny `EPSILON` constant and compute a “lift” ratio for each metric
    `(post_avg + ε)/(pre_avg + ε)`;
* split the resulting author summary into juniors (treatment = 1) and seniors (0) and
    inspect median lifts;
* merge the lift values back onto the matched‑pair table so that each treated author is
    paired with their control, then run paired Wilcoxon signed‑rank tests on the lifts
    for citations and works, reporting the statistic, p‑value, and median difference;
* finally, persist the full author lift table and the paired table to CSV, and export a
    small summary table used for later figures.

This notebook is therefore a complete preprocessing and comparison pipeline: loading the
data, transforming it into a suitable form, computing before/after averages and lift
ratios, comparing juniors versus seniors via matched pairs, and saving the results for
downstream analysis.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

ROOT          = Path("..")
DATA_MATCHED  = ROOT / "data" / "matched"
DATA_PROFILES = ROOT / "data" / "profiles"
DATA_FIGURES  = ROOT / "data" / "figures"

In [2]:
pairs    = pd.read_csv(DATA_MATCHED  / "matched_pairs.csv")
profiles = pd.read_csv(DATA_PROFILES / "all_matched_profiles.csv")

print("matched_pairs:       ", pairs.shape)
print("all_matched_profiles:", profiles.shape)
print("\nprofiles sample:")
print(profiles.head(3))

matched_pairs:        (603, 10)
all_matched_profiles: (1198, 8)

profiles sample:
                          author_id       author_name  works_count  \
0  https://openalex.org/A5014823249      Jincheng Mei           29   
1  https://openalex.org/A5069646529  Adhiguna Kuncoro           35   
2  https://openalex.org/A5032195981      Sang-Gyun An            8   

   cited_by_count                                     counts_by_year  \
0             244  [{"year": 2014, "works_count": 3, "oa_works_co...   
1            1578  [{"year": 2016, "works_count": 4, "oa_works_co...   
2             131  [{"year": 2017, "works_count": 2, "oa_works_co...   

   award_year conference  treatment  
0        2018       AAAI          1  
1        2018        ACL          1  
2        2018        CHI          1  


In [3]:
import ast

def parse_counts(row):
    """Expand the counts_by_year JSON list into one row per year."""
    try:
        entries = ast.literal_eval(row["counts_by_year"])
    except Exception:
        return pd.DataFrame()
    rows = []
    for e in entries:
        rows.append({
            "author_id"  : row["author_id"],
            "treatment"  : row["treatment"],
            "conference" : row["conference"],
            "award_year" : row["award_year"],
            "cal_year"   : int(e["year"]),
            "works"      : int(e.get("works_count", 0)),
            "citations"  : int(e.get("cited_by_count", 0)),
        })
    return pd.DataFrame(rows)

long = pd.concat(
    [parse_counts(r) for _, r in profiles.iterrows()],
    ignore_index=True
)

# Compute relative year
long["rel_year"] = long["cal_year"] - long["award_year"]

# Keep ±10 window
long = long[long["rel_year"].between(-10, 10)]

print("Long format shape:", long.shape)
print("rel_year range:", long["rel_year"].min(), "→", long["rel_year"].max())
print(long.head())


Long format shape: (13912, 8)
rel_year range: -10 → 10
                          author_id  treatment conference  award_year  \
0  https://openalex.org/A5014823249          1       AAAI        2018   
1  https://openalex.org/A5014823249          1       AAAI        2018   
2  https://openalex.org/A5014823249          1       AAAI        2018   
3  https://openalex.org/A5014823249          1       AAAI        2018   
4  https://openalex.org/A5014823249          1       AAAI        2018   

   cal_year  works  citations  rel_year  
0      2014      3         46        -4  
1      2015      1          2        -3  
2      2016      3         10        -2  
3      2017      1         21        -1  
4      2018      2         20         0  


In [4]:
PRE_WINDOW  = range(-5, 0)   # rel_year -5 to -1
POST_WINDOW = range(1, 6)    # rel_year +1 to +5

def window_avg(df, author_id, window, metric):
    sub = df[(df["author_id"] == author_id) & (df["rel_year"].isin(window))]
    return sub[metric].mean() if len(sub) > 0 else np.nan

# Build per-author summary
authors = profiles[["author_id", "author_name", "treatment", "conference", "award_year"]].copy()

for metric in ["citations", "works"]:
    authors[f"pre_avg_{metric}"]  = authors["author_id"].apply(
        lambda aid: window_avg(long, aid, PRE_WINDOW, metric))
    authors[f"post_avg_{metric}"] = authors["author_id"].apply(
        lambda aid: window_avg(long, aid, POST_WINDOW, metric))

print("Author summary shape:", authors.shape)
print(authors[["author_name","treatment","pre_avg_citations","post_avg_citations"]].head(6))

Author summary shape: (1198, 9)
        author_name  treatment  pre_avg_citations  post_avg_citations
0      Jincheng Mei          1              19.75                48.0
1  Adhiguna Kuncoro          1             319.00               145.6
2      Sang-Gyun An          1              27.00                15.0
3    Sylvia Simioni          1                NaN                36.0
4      Sean Peacock          1               2.50                17.5
5  Clara Crivellaro          1             141.50                63.4


In [5]:
EPSILON = 1e-6  # avoid division by zero

for metric in ["citations", "works"]:
    authors[f"lift_{metric}"] = (
        (authors[f"post_avg_{metric}"] + EPSILON) /
        (authors[f"pre_avg_{metric}"]  + EPSILON)
    )

# Split
juniors = authors[authors["treatment"] == 1].copy()
seniors = authors[authors["treatment"] == 0].copy()

print(f"Juniors: {len(juniors)} | Seniors: {len(seniors)}")
print("\n── Median Lift (Citations) ──")
print(f"  Junior:  {juniors['lift_citations'].median():.3f}")
print(f"  Senior:  {seniors['lift_citations'].median():.3f}")
print("\n── Median Lift (Works) ──")
print(f"  Junior:  {juniors['lift_works'].median():.3f}")
print(f"  Senior:  {seniors['lift_works'].median():.3f}")

Juniors: 603 | Seniors: 595

── Median Lift (Citations) ──
  Junior:  1.660
  Senior:  1.628

── Median Lift (Works) ──
  Junior:  1.562
  Senior:  1.715


#### Wilcoxon Signed-Rank Test

In [6]:
# Merge on matched pairs to get proper paired comparison
pairs_lift = pairs[["treated_id", "control_id", "conference", "award_year", "core_rank"]].merge(
    authors[["author_id", "lift_citations", "lift_works"]].rename(
        columns={"author_id": "treated_id",
                 "lift_citations": "lift_cit_treated",
                 "lift_works": "lift_works_treated"}),
    on="treated_id", how="left"
).merge(
    authors[["author_id", "lift_citations", "lift_works"]].rename(
        columns={"author_id": "control_id",
                 "lift_citations": "lift_cit_control",
                 "lift_works": "lift_works_control"}),
    on="control_id", how="left"
).dropna(subset=["lift_cit_treated", "lift_cit_control"])

print(f"Valid pairs for test: {len(pairs_lift)}")

for metric, t_col, c_col in [
    ("Citations", "lift_cit_treated",   "lift_cit_control"),
    ("Works",     "lift_works_treated", "lift_works_control"),
]:
    stat, p = stats.wilcoxon(pairs_lift[t_col], pairs_lift[c_col])
    median_diff = (pairs_lift[t_col] - pairs_lift[c_col]).median()
    print(f"\n── Wilcoxon: {metric} ──")
    print(f"  Median lift difference (junior − senior): {median_diff:.3f}")
    print(f"  Statistic: {stat:.1f} | p-value: {p:.4f}")
    print(f"  Significant at 0.05: {'✓ YES' if p < 0.05 else '✗ NO'}")

Valid pairs for test: 3592

── Wilcoxon: Citations ──
  Median lift difference (junior − senior): -0.206
  Statistic: 3172576.0 | p-value: 0.3855
  Significant at 0.05: ✗ NO

── Wilcoxon: Works ──
  Median lift difference (junior − senior): -0.180
  Statistic: 2836613.0 | p-value: 0.0000
  Significant at 0.05: ✓ YES


In [8]:
# Save full author lift table
authors.to_csv(DATA_MATCHED / "author_lift.csv", index=False)
print("✓ author_lift.csv saved →", DATA_MATCHED / "author_lift.csv")

# Save paired lift table (for forest plot in Day 5)
pairs_lift.to_csv(DATA_MATCHED / "pairs_lift.csv", index=False)
print("✓ pairs_lift.csv saved →", DATA_MATCHED / "pairs_lift.csv")

# Quick summary table
summary = authors.groupby("treatment").agg(
    n                    = ("author_id", "count"),
    median_lift_cit      = ("lift_citations", "median"),
    median_lift_works    = ("lift_works", "median"),
    mean_pre_cit         = ("pre_avg_citations", "mean"),
    mean_post_cit        = ("post_avg_citations", "mean"),
).reset_index()
summary["treatment"] = summary["treatment"].map({1: "Junior", 0: "Senior"})
print("\n── Summary Table ──")
print(summary.to_string(index=False))
summary.to_csv(DATA_FIGURES / "lift_summary.csv", index=False)
print("✓ lift_summary.csv saved")

✓ author_lift.csv saved → ..\data\matched\author_lift.csv
✓ pairs_lift.csv saved → ..\data\matched\pairs_lift.csv

── Summary Table ──
treatment   n  median_lift_cit  median_lift_works  mean_pre_cit  mean_post_cit
   Senior 595         1.628235           1.715443    151.460830     276.060383
   Junior 603         1.659772           1.562500    146.271773     226.378835
✓ lift_summary.csv saved
